In [1]:
# ==========================================================
# HEALTH INSURANCE CLAIM PREDICTION - RANDOM FOREST MODEL
# ==========================================================

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.ensemble import RandomForestRegressor
import pickle

print("Loading dataset...")

df = pd.read_csv("insurance_data.csv")

print("Original Dataset Shape:", df.shape)

# ------------------------------------------------
# Drop ID column
# ------------------------------------------------
df = df.drop("PatientID", axis=1)

# ------------------------------------------------
# Handle NULL values
# ------------------------------------------------
for col in df.columns:
    
    if df[col].dtype == "object":
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].mean())

print("Null values handled")

# ------------------------------------------------
# Encode categorical variables
# ------------------------------------------------
le = LabelEncoder()

df["gender"] = le.fit_transform(df["gender"])
df["smoker"] = le.fit_transform(df["smoker"])
df["region"] = le.fit_transform(df["region"])
df["diabetic"] = le.fit_transform(df["diabetic"])

# ------------------------------------------------
# Feature Engineering
# ------------------------------------------------
df["age_bmi"] = df["age"] * df["bmi"]
df["bmi_bp"] = df["bmi"] * df["bloodpressure"]
df["age_smoker"] = df["age"] * df["smoker"]

# ------------------------------------------------
# Remove extreme outliers
# ------------------------------------------------
q_low = df["claim"].quantile(0.01)
q_high = df["claim"].quantile(0.99)

df = df[(df["claim"] > q_low) & (df["claim"] < q_high)]

print("Dataset after cleaning:", df.shape)

# ------------------------------------------------
# Split features and target
# ------------------------------------------------
X = df.drop("claim", axis=1)
y = df["claim"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Training Random Forest model...")

# ------------------------------------------------
# Random Forest Model
# ------------------------------------------------
model = RandomForestRegressor(
    n_estimators=1000,
    max_depth=12,
    min_samples_split=5,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# ------------------------------------------------
# Prediction
# ------------------------------------------------
y_pred = model.predict(X_test)

r2 = r2_score(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("\nMODEL RESULTS")
print("R2 Score:", round(r2,3))
print("MAE:", round(mae,2))
print("RMSE:", round(rmse,2))

# ------------------------------------------------
# Save model
# ------------------------------------------------
model_data = {
    "model": model
}

with open("insurance_model.pkl", "wb") as f:
    pickle.dump(model_data, f)

print("\nModel saved successfully: insurance_model.pkl")

Loading dataset...
Original Dataset Shape: (1340, 10)
Null values handled
Dataset after cleaning: (1312, 12)
Training Random Forest model...

MODEL RESULTS
R2 Score: 0.836
MAE: 3576.27
RMSE: 4703.48

Model saved successfully: insurance_model.pkl


In [6]:
df

,age,gender,bmi,bloodpressure,diabetic,children,smoker,region,claim,age_bmi,bmi_bp,age_smoker
13,32.0,1,27.6,100,0,0,0,2,1252.41,883.2,2760.0,0.0
14,40.0,1,28.7,81,1,0,0,2,1253.94,1148.0,2324.7,0.0
15,32.0,1,30.4,86,1,0,0,2,1256.30,972.8,2614.4,0.0
16,35.0,1,34.1,90,0,0,0,3,1261.44,1193.5,3069.0,0.0
17,41.0,1,34.4,84,0,0,0,3,1261.86,1410.4,2889.6,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...
1321,26.0,1,37.0,81,0,2,1,1,47496.49,962.0,2997.0,26.0
1322,33.0,0,36.8,117,1,1,1,0,47896.79,1214.4,4305.6,33.0
1323,49.0,0,33.8,107,0,1,1,3,47928.03,1656.2,3616.6,49.0
1324,39.0,1,39.9,115,0,0,1,3,48173.36,1556.1,4588.5,39.0


In [7]:
df.columns

Index(['age', 'gender', 'bmi', 'bloodpressure', 'diabetic', 'children',
       'smoker', 'region', 'claim', 'age_bmi', 'bmi_bp', 'age_smoker'],
      dtype='object')